In [4]:
# ============================================================
# ML MODEL 2: RANDOM FOREST WITH PREPROCESSING
# ============================================================

# Import required libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. SELECT FEATURES AND TARGET
# ------------------------------------------------------------

# Use the same features as our Logistic Regression model.
# failure_count is excluded because it was used to create the target.

features = [
    "department_id",
    "equipment_type",
    "equipment_age_days",
    "days_until_next_maintenance",
    "usage_hours",
    "maintenance_count"
]

X = equipment[features]
y = equipment["failure_risk"]


# ------------------------------------------------------------
# 2. DEFINE FEATURE TYPES
# ------------------------------------------------------------

categorical_features = [
    "department_id",
    "equipment_type"
]

numerical_features = [
    "equipment_age_days",
    "days_until_next_maintenance",
    "usage_hours",
    "maintenance_count"
]


# ------------------------------------------------------------
# 3. PREPROCESS THE DATA
# ------------------------------------------------------------

# Numerical columns:
# Fill missing values using the median.

# Categorical columns:
# Fill missing values using the most frequent value,
# then convert categories into numerical columns.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            SimpleImputer(strategy="median"),
            numerical_features
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# 4. TRAIN / TEST SPLIT
# ------------------------------------------------------------

# Keep the same 80/20 split strategy used for Model 1.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


# ------------------------------------------------------------
# 5. CREATE RANDOM FOREST PIPELINE
# ------------------------------------------------------------

# The pipeline first converts text categories into numbers,
# then trains the Random Forest model.

rf_model = Pipeline([
    (
        "preprocessing",
        preprocessor
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
    )
])


# ------------------------------------------------------------
# 6. TRAIN MODEL
# ------------------------------------------------------------

rf_model.fit(X_train, y_train)

print("\nRandom Forest training completed successfully!")


# ------------------------------------------------------------
# 7. MAKE PREDICTIONS
# ------------------------------------------------------------

rf_pred = rf_model.predict(X_test)

print("Predictions generated successfully!")


# ------------------------------------------------------------
# 8. EVALUATE MODEL
# ------------------------------------------------------------

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)

print("\n========== RANDOM FOREST PERFORMANCE ==========")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))


# ------------------------------------------------------------
# 9. CONFUSION MATRIX
# ------------------------------------------------------------

print("\n========== CONFUSION MATRIX ==========")
print(confusion_matrix(y_test, rf_pred))


# ------------------------------------------------------------
# 10. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_test,
        rf_pred,
        target_names=[
            "No Failure",
            "Failure Risk"
        ]
    )
)


# Save the trained Random Forest pipeline.
# The saved file will later be loaded by Streamlit to predict
# equipment failure risk for new hospital equipment data.

import joblib
from pathlib import Path

model_path = Path("../ml/models/equipment_failure_model.pkl")

# Create the models folder if it does not already exist.
model_path.parent.mkdir(parents=True, exist_ok=True)

# Save the complete pipeline, including preprocessing and Random Forest.
joblib.dump(rf_model, model_path)

print("Equipment failure model saved successfully!")
print("Saved at:", model_path)

Training data: (4000, 6)
Testing data: (1000, 6)

Random Forest training completed successfully!
Predictions generated successfully!

========== RANDOM FOREST PERFORMANCE ==========
Accuracy : 0.785
Precision: 0.8079
Recall   : 0.8313
F1 Score : 0.8195

========== CONFUSION MATRIX ==========
[[297 116]
 [ 99 488]]

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

  No Failure       0.75      0.72      0.73       413
Failure Risk       0.81      0.83      0.82       587

    accuracy                           0.79      1000
   macro avg       0.78      0.78      0.78      1000
weighted avg       0.78      0.79      0.78      1000

Equipment failure model saved successfully!
Saved at: ..\ml\models\equipment_failure_model.pkl
